# Example 16.4 / 1: Model a Self-Describing Sequence

Companion notebook for **16.4 Magic Sequences** in the AMPL lab Documentation, section "Model a Self-Describing Sequence". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Model a Self-Describing Sequence

`exactly k {j in S} (condition)` is true when the condition holds for exactly `k` members of `S`. In `Describe` below, `k` is itself a decision variable `s[i]`, and the condition counts the positions whose entry equals `i`. There is no objective because any sequence satisfying all `Describe` constraints answers the puzzle.

For example, $(1,2,1,0)$ is magic for $n=4$: zero appears once, one twice, two once, and three zero times. $(2,1,0,1)$ is not: it claims zero appears twice, but zero appears once.

The run solves $n=4$ with HiGHS, reads `s` with `to_list()`, and checks the result with an ordinary frequency count in Python:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param n integer > 0;  # sequence length
set POS := 0..n-1;  # positions, which are also the counted values
var s {POS} integer >= 0, <= n-1;  # s[i] = number of entries equal to i

subject to Describe {i in POS}:  # s[i] counts the occurrences of i
    exactly s[i] {j in POS} (s[j] = i);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["n"] = 4

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
seq = [round(v) for i, v in ampl.var["s"].to_list()]
print("sequence:", seq)
print("independent check:", all(seq.count(i) == seq[i] for i in range(len(seq))))

solved
sequence: [2, 0, 2, 0]
independent check: True


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param n := 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
solve;
display solve_result, s;

solve_result = solved

s [*] :=
0  2
1  0
2  2
3  0
;



### Another way in amplpy

To list every magic sequence of a length, record each one in `Found` and add the no-good cut `Exclude` from N-Queens (AMPL lab documentation, section 16.1): every new sequence must differ from each recorded one in at least one position. The run adds these lines with `ampl.eval`, because amplpy has no method that declares new model components, then repeats solve, record and exclude until `ampl.solve_result` is not `solved`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["n"] = 4

In [11]:
# run.py
ampl.eval("""
param nfound integer >= 0 default 0;  # sequences recorded so far
param Found {1..nfound, POS};  # entries of each recorded sequence
subject to Exclude {k in 1..nfound}:  # no-good cut: differ from every recorded sequence
    exists {i in POS} s[i] != Found[k,i];
""")
for n in [3, 4, 5, 6]:
    ampl.param["n"] = n
    ampl.param["nfound"] = 0
    ampl.eval("reset data Found;")
    found = []
    while True:
        ampl.solve(solver="highs", verbose=False)
        if ampl.solve_result != "solved":
            break
        seq = [round(v) for i, v in ampl.var["s"].to_list()]
        found.append(seq)
        ampl.param["nfound"] = len(found)
        for i, v in enumerate(seq):
            ampl.param["Found"][len(found), i] = v
    print(f"n = {n}: {found}, final status {ampl.solve_result}")

n = 3: [], final status infeasible
n = 4: [[2, 0, 2, 0], [1, 2, 1, 0]], final status infeasible
n = 5: [[2, 1, 2, 0, 0]], final status infeasible
n = 6: [], final status infeasible


### Pure AMPL: data.dat and model.run

The same for $n=4$ in pure AMPL, with the same `model.mod`:

In [12]:
%%writefile data_2.dat
param n := 4;

Writing data_2.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data_2.dat;

In [14]:
%%ampl_eval
# model.run
param nfound integer >= 0 default 0;  # sequences recorded so far
param Found {1..nfound, POS};  # entries of each recorded sequence
subject to Exclude {k in 1..nfound}:  # no-good cut: differ from every recorded sequence
    exists {i in POS} s[i] != Found[k,i];

option solver highs, solver_msg 0;
repeat {
    solve;
    if solve_result <> "solved" then break;
    let nfound := nfound + 1;
    let {i in POS} Found[nfound,i] := round(s[i]);
    printf "sequence %d:", nfound;
    printf {i in POS} " %d", round(s[i]);
    printf "\n";
};
display nfound, solve_result;

sequence 1: 2 0 2 0
sequence 2: 1 2 1 0
nfound = 2
solve_result = infeasible



### Another way in amplpy

Only $n=4$ has two sequences among these sizes, $(2,0,2,0)$ and $(1,2,1,0)$; $n=3$ and $n=6$ end infeasible without any solution. For $n=5$, the second solve excludes $(2,1,2,0,0)$ and is `infeasible`, which proves that this sequence is the only one of length 5.

**Strengthening equations.** Every feasible magic sequence also satisfies

$$
\sum_{i=0}^{n-1}s_i=n,
\qquad
\sum_{i=0}^{n-1}i\,s_i=n.
$$

The first equation counts the sequence's $n$ positions by value. The second sums all values in two ways: directly through the entries and through their occurrence counts. They are logically redundant with a correct direct model, but can strengthen a reformulation and expose bad candidate data.

Subtracting the first identity from the second gives another useful check:

$$
s_0=\sum_{i=2}^{n-1}(i-1)s_i.
$$

It says that the number of zero entries balances the excess contributed by values two and above. For $(1,2,1,0)$ it reads $1=(2-1)\cdot1+(3-1)\cdot0$. This relation is necessary, but the two sums are not sufficient: the non-magic $(2,1,0,1)$ satisfies both, since $2+1+0+1=4$ and $0\cdot2+1\cdot1+2\cdot0+3\cdot1=4$.

The run below solves $n=10$ with the direct model, adds the two equations with `ampl.eval`, and solves again:

In [15]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [16]:
# data.py
ampl.param["n"] = 4

In [17]:
# run.py
ampl.param["n"] = 10
ampl.solve(solver="highs", verbose=False)
print("direct:", [round(v) for i, v in ampl.var["s"].to_list()])
ampl.eval("""
subject to CountPositions:  # redundant: the counts cover all n positions
    sum {i in POS} s[i] = n;
subject to CountValues:  # redundant: the entries sum to n
    sum {i in POS} i * s[i] = n;
""")
ampl.solve(solver="highs", verbose=False)
print("strengthened:", [round(v) for i, v in ampl.var["s"].to_list()])

direct: [6, 2, 1, 0, 0, 0, 1, 0, 0, 0]
strengthened: [6, 2, 1, 0, 0, 0, 1, 0, 0, 0]
